# 1. 闭包 


## 1.1 嵌套函数（闭包的前提）  


In [28]:
def outer():
    def inner():
        print("This is the inner function.")
    return inner                #返回的是引用

outer()            #调用outer函数，返回inner函数的引用
outer()()          #调用outer函数，返回inner函数的引用，再调用inner函数     

a = outer()        #调用outer函数，返回inner函数的引用，并将其赋值给变量a
a()                #调用变量a，实际上是在调用inner函数

This is the inner function.
This is the inner function.


## 1.2 什么是闭包函数？
闭包就像“带记忆的工具”--外层函数定义了一个变量，内层函数使用了这个变量，而且外层函数把内层函数“打包返回”。  
哪怕外层函数执行完了，内层函数依然能“记住”那个变量的值。

## 1.3 闭包函数的3个构成条件  
- **函数嵌套**：外层函数定义内层函数；
- **变量引用**：内层函数使用外层函数的局部变量（不是全局变量，也不是内层自己的变量）；
- **返回内层函数**：外层函数的返回值是内层函数的引用（不加括号，只返回函数名）。


In [29]:
# 升级版闭包，让外层函数接收参数，动态记忆不同的值
def outer(base_num):
    def inner(add_num):
        print(f"{base_num} + {add_num} = {base_num + add_num}")
    return inner

add_five = outer(5)    #创建一个闭包，记忆base_num的值为5
add_five(10)           #调用闭包，传入add_num的值为

add_ten = outer(10)   #创建另一个闭包，记忆base_num的值为10
add_ten(20)            #调用闭包，传入add_num的值为


5 + 10 = 15
10 + 20 = 30


## 1.4闭包的核心作用
1.**保存状态**：让函数记住关键变量，避免重复传入相同参数。  
2.**数据封装**：外层变量只能被内层函数访问，外部无法直接修改，保证数据安全。  
3.**基础铺垫**：是装饰器的核心实现原理，学会闭包，装饰器就简单了。   

---

# 2. 装饰器：不修改原函数，添加新功能  
## 2.1 为什么需要装饰器？ 
场景:你写了3个函数(发消息、转账、看动态)，现在要给每个函数加“登录验证”功能一如果直接修改每个函数的代码，会导致代码冗余;如果函数是第三方库的(不能修改)，就完全没办法。   
装饰器的解决方案:**像给函数“穿衣服”一样，给原函数套一层新功能，原函数本身不变**  既不修改代码，又不改变调用方式。


## 2.2 装饰器的本质  
装饰器本质是一个"接收函数作为参数、返回新函数的闭包”--输入是原函数，输出是“原函数+新功能”的组合函数。
示例:给“发消息、转账”函数加“登录验证”功能(基础版装饰器，给函数加固定功能)

In [30]:
def login_decorator(func):
    def wrapper():
        print("Checking login status...")
        # 假设用户已登录，直接调用原函数
        func()
    return wrapper

def send_msg():
    print("Send message function.")

def send_money():
    print("Send money function.")

def look_news():
    print("Look news function.")

login_send_msg = login_decorator(send_msg)  #创建一个装饰器，包装send_msg函数
login_send_msg()  #调用装饰器，实际上是在调用wrapper函数


Checking login status...
Send message function.


#### 手动包装太麻烦，使用`@装饰器名`的语法糖，直接放在函数上面即可完成包装

In [31]:
def login_decorator(func):
    def wrapper():
        print("Checking login status...")
        # 假设用户已登录，直接调用原函数
        func()
    return wrapper

@login_decorator
def send_msg():
    print("Send message function.")

@login_decorator
def send_money():
    print("Send money function.")

@login_decorator
def look_news():
    print("Look news function.")

send_msg()    #调用被装饰的函数，实际上是在调用wrapper函数
send_money()  #调用被装饰的函数，实际上是在调用wrapper函数
look_news()   #调用被装饰的函数，实际上是在调用wrapper函数



Checking login status...
Send message function.
Checking login status...
Send money function.
Checking login status...
Look news function.


#### 关键注意
如果原函数需要接收参数(如“发消息给xxx”)，装饰器的内层函数要支持参数传递，用`*args`和`**kwargs`  
(接收任意数量、任意类型的参数)

In [32]:
def login_decorator(func):
    def wrapper(*args,**kwargs):
        print("Checking login status...")
        # 假设用户已登录，直接调用原函数
        func(*args, **kwargs)
    return wrapper

@login_decorator
def send_msg(name):
    print(f"Send message to {name}.")

@login_decorator
def send_money(name,money=100):
    print(f"Send {money} dollars to {name}.")


send_msg("Alice")          #调用被装饰的函数，实际上是在调用wrapper函数
send_money("Bob")          #调用被装饰的函数，实际上是在调用wrapper函数


Checking login status...
Send message to Alice.
Checking login status...
Send 100 dollars to Bob.


#### 实用技巧
一个函数可以套多个装饰器，执行顺序是:离函数最近的装饰器先执行，从内到外依次叠加(像穿衣:先穿内衣，再穿外套)。

## 2.3 常见应用场景

1. **登录验证**  
   需要登录才能执行函数（如评论、下单、转账）。

2. **日志记录**  
   记录函数的执行时间、参数、返回结果（方便调试）。

3. **性能监测**  
   统计函数的执行耗时。

4. **权限控制**  
   判断用户是否有执行函数的权限（如管理员才能删除数据）。

## 2.4 核心总结
1. 闭包

- **本质**：带“记忆功能”的嵌套函数；
- **3 个条件**：
  - 函数嵌套；
  - 内层函数使用外层变量；
  - 外层函数返回内层函数；
- **作用**：保存状态、数据封装、支撑装饰器。


2. 装饰器

- **本质**：接收函数，返回新函数的闭包；
- **核心价值**：不修改原函数，给函数添加额外功能；

### 关键语法

- **基础装饰器**
```python
def decorator(func):
    def wrapper(*args, **kwargs):
        # 新功能
        result = func(*args, **kwargs)
        return result
    return wrapper